# 02 - SFT (Supervised Fine-Tuning) Eğitimi

**Amaç:** Taban model `Qwen/Qwen2.5-0.5B` üzerine, `HuggingFaceH4/no_robots`
gösterim veri kümesi (Rajani vd., 2023) ile LoRA (Hu vd., 2022) kullanılarak
asistan personası kazandırmak ve **Model_SFT**'yi üretmek.

- Yazılım altyapısı: `trl.SFTTrainer`, `peft.LoraConfig`, `bitsandbytes` (4-bit QLoRA)
- Çıktı: `outputs/sft/adapter` (LoRA adaptörü) ve `outputs/sft/merged_model`
  (adaptörün taban modele gömüldüğü tam checkpoint - **Model_SFT**)

> Bu defter Google Colab Pro (A100/L4 GPU) üzerinde çalıştırılmak üzere tasarlanmıştır.

## 0. Ortam Kurulumu

In [ ]:
# --- Google Colab ortam kurulumu -------------------------------------------
# Bu hucre, projenin tamamini (Datasets/, src/, notebooks/, outputs/) iceren
# "Uygulama" klasorunu Google Drive'a yukledikten sonra calistirilmalidir.
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # TODO: Kendi Google Drive yolunuza gore guncelleyin
    PROJECT_ROOT = "/content/drive/MyDrive/Uygulama"
else:
    PROJECT_ROOT = os.path.abspath("..")

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IN_COLAB:", IN_COLAB)

In [ ]:
# NOT: Surumler, TRL'nin klasik `PPOTrainer` + `AutoModelForCausalLMWithValueHead`
# coklu-adaptor (multi-adapter) RL ozelligiyle dogrulanmis, bilinen-uyumlu bir
# kombinasyon olacak sekilde SABITLENMISTIR (API kararliligi icin).
%pip install -q -U "transformers==4.44.2" "trl==0.9.6" "peft==0.11.1" "bitsandbytes==0.43.1" "accelerate==0.32.1" "datasets==2.20.0" "numpy==1.26.4" sentencepiece

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import alignment_pipeline as ap

ap.set_global_seed(42)
print("Cihaz:", ap.get_device())

In [ ]:
config = ap.ExperimentConfig(
    dataset_root=os.path.join(PROJECT_ROOT, "Datasets"),
    output_root=os.path.join(PROJECT_ROOT, "outputs"),
)
ap.ensure_dir(config.output_root)
print("Taban model:", config.base_model_id)
print("Veri kok dizini:", config.dataset_root)
print("Cikti kok dizini:", config.output_root)

In [ ]:
SFT_DIR = config.path("sft")
SFT_ADAPTER_DIR = os.path.join(SFT_DIR, "adapter")
SFT_MERGED_DIR = os.path.join(SFT_DIR, "merged_model")
SFT_LOGS_DIR = os.path.join(SFT_DIR, "logs")
for d in (SFT_ADAPTER_DIR, SFT_MERGED_DIR, SFT_LOGS_DIR):
    ap.ensure_dir(d)

# A100 (80GB) üzerinde 0.5B model için nicemleme (quantization) zorunlu değildir;
# ancak metodolojik tutarlılık (bitsandbytes kullanımı) için varsayılan olarak
# 4-bit QLoRA stili yükleme kullanılmıştır. Daha küçük GPU'larda da bellek
# avantajı sağlar. Tam hassasiyet isterseniz `QUANT_BITS = None` yapın.
QUANT_BITS = 4

## 1. Model, Tokenizer ve LoRA Yapılandırması

In [ ]:
tokenizer = ap.load_tokenizer(config.base_model_id)
sft_lora_config = ap.make_lora_config(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=ap.QWEN2_LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
)
print(sft_lora_config)

## 2. Veri Hazırlığı

In [ ]:
no_robots = ap.load_no_robots_sft(config.dataset_root)
train_dataset = ap.add_chat_text_column(no_robots["train"], tokenizer)
eval_dataset = ap.add_chat_text_column(no_robots["test"], tokenizer)

print("Eğitim örnek sayısı:", len(train_dataset))
print("Doğrulama örnek sayısı:", len(eval_dataset))
print()
print(train_dataset[0]["text"][:500])

## 3. SFT Eğitimi

> **Not (hızlı deneme):** Kavram-kanıtlama (PoC) sırasında hızlı yineleme için
> `train_dataset.select(range(N))` ile küçük bir alt küme kullanabilirsiniz.

In [ ]:
trainer, sft_model = ap.run_sft(
    model_id=config.base_model_id,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    lora_config=sft_lora_config,
    output_dir=SFT_DIR,
    text_field="text",
    max_seq_length=config.max_seq_length,
    num_train_epochs=3.0,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    quant_bits=QUANT_BITS,
)

## 4. Eğitim Eğrisi (Nicel)

In [ ]:
log_df = ap.trainer_log_history_to_df(trainer)
ap.save_json(trainer.state.log_history, os.path.join(SFT_LOGS_DIR, "sft_log_history.json"))
ap.plot_loss_curve(log_df, loss_col="loss", title="SFT Eğitim Kaybı (Model_SFT)")

In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)
ap.save_json(eval_metrics, os.path.join(SFT_LOGS_DIR, "sft_eval_metrics.json"))

## 5. Adaptörü Kaydet ve Taban Modele Göm (Merge) → Model_SFT

RM / PPO / DPO adımlarının standart bir hizalama hattında (pretrain → SFT →
RM → RL/DPO) olduğu gibi **Model_SFT** üzerinden devam edebilmesi için, SFT
LoRA adaptörü taban model ağırlıklarına gömülüp tam bir checkpoint olarak
kaydedilir.

In [ ]:
sft_model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)

ap.merge_lora_adapter_and_save(config.base_model_id, SFT_ADAPTER_DIR, SFT_MERGED_DIR)
print("Model_SFT (merged) kaydedildi ->", SFT_MERGED_DIR)

## 6. Hızlı Nitel Kontrol: Model_Base vs. Model_SFT

Eğitim öncesi/sonrası birkaç örnek istem üzerinde modelin asistan personası
kazanıp kazanmadığı hızlıca gözlemlenir (ayrıntılı nitel karşılaştırma
`06_Algorithm_Comparison_Qualitative.ipynb` içinde yapılacaktır).

In [ ]:
test_prompts = ap.DEFAULT_QUALITATIVE_TEST_PROMPTS[:3]

base_model, base_tokenizer = ap.load_model_for_inference(config.base_model_id, adapter_path=None, quant_bits=QUANT_BITS)
sft_model_inf, sft_tokenizer = ap.load_model_for_inference(config.base_model_id, adapter_path=SFT_ADAPTER_DIR, quant_bits=QUANT_BITS)

base_responses = ap.generate_batch(base_model, base_tokenizer, test_prompts, max_new_tokens=128)
sft_responses = ap.generate_batch(sft_model_inf, sft_tokenizer, test_prompts, max_new_tokens=128)

for p, b, s in zip(test_prompts, base_responses, sft_responses):
    print("İSTEM:", p)
    print("Model_Base:", b[:300])
    print("Model_SFT :", s[:300])
    print("-" * 80)

## 7. Sonuç

`Model_SFT` (`outputs/sft/merged_model`) başarıyla üretildi. Sıradaki adım:
`03_RLHF_Training.ipynb` (Ödül Modeli + çoklu-adaptör PPO).